## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_Q(data, fn):
    """get heat flux over region using specified fn"""

    ### reconstruct flux over region
    hf = src.utils.reconstruct_wrapper(data, fn=fn)

    ## convert units to K/yr
    sec_per_mo = 8.64e4 * 30
    sec_per_yr = 12 * sec_per_mo
    rho = 1.02e3
    Cp = 4.2e3
    H = 50
    # H = 80

    ## apply scaling fac.
    Q = hf * sec_per_yr / (rho * Cp * H)

    return Q

def sel_eq(data):
    """select equatorial region"""
    idx = dict(latitude=slice(-5,5), longitude=slice(120,280))
    return data.sel(idx).mean("latitude")

## Load data

### surface heat fluxes

In [ ]:
flux_forced, flux_anom = src.utils.load_flux_data()

## switch sign convection to be positive into the ocean
for v in ["flns", "lhflx"]:
    flux_anom[v] = -1 * flux_anom[v]
    flux_forced[v] = -1 * flux_forced[v]

## load into memory
flux_anom.load();

### $T$, $h$

In [ ]:
Th = src.utils.load_cesm_indices().compute()

In [ ]:
T = xr.merge(
    [(Th["T_3"]**i).rename(f"T{i}") for i in range(4)]
)

### merge

In [ ]:
data = xr.merge(
    [
        T, 
        flux_anom[[v for v in list(flux_anom) if "comp" not in v]]
    ]
)

## window in time
data = src.utils.get_windowed(data, stride=120, window_size=480)
data = data.sel(year=slice(None,2080))

## fit

In [ ]:
def regress_pinv(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    # prep = lambda x: x.transpose("member", ...)
    prep = lambda x: x.stack(s=["time", "member"]).transpose("year",..., "s")

    X_ = prep(X[x_vars].to_dataarray(dim="v"))
    Y_ = prep(X[y_var]).transpose("year","mode",...)

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year": X.year, "mode": Y_.mode, "v": x_vars},
        dims=["year", "mode", "v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("...ki,...ij->...kj", Y_.values, X_pinv)

    return m.rename({"v": "param"})


def regress_pinv_wrapper(X, x_vars, y_var):
    return X.groupby("time.month").map(regress_pinv, x_vars=x_vars, y_var=y_var)

In [ ]:
flux_types=["fsns", "lhflx", "flns"]

## empty array to hold results
alpha = []

## loop thru flux types
for ftype in flux_types:

    kwargs = dict(x_vars=["T0", "T1", "T2", "T3"], y_var=ftype)
    alpha.append(-regress_pinv_wrapper(data, **kwargs))

## merge
alpha = xr.merge([a.rename(f) for a, f in zip(alpha, flux_types)])

## add components
alpha = xr.merge([alpha, flux_anom[[f"{f}_comp" for f in flux_types]]])

## compute to heat flux
Q = get_Q(alpha, fn=sel_eq)

## plot results

### timeseries

In [ ]:
sel_ = lambda x : x.sel(param="T1", longitude=slice(210,270), month=slice(6,12)).mean(["longitude","month"])
delta = lambda x : sel_(x)-sel_(x).isel(year=0)

fig,ax = plt.subplots(figsize=(4,3))
for f in flux_types:
    ax.plot(Q.year, delta(Q[f]), label=f)

ax.legend()

plt.show()

### Plot spatial structure

In [ ]:
sel_ = lambda x : x.sel(month=slice(6,12)).mean(["month"])

fig,axs = plt.subplots(1,4,figsize=(10,2.5), layout="constrained")

for ax, p in zip(axs, Q.param.values):

    ax.plot(Q.longitude, sel_(Q["fsns"]).sel(year=1870, param=p))
    ax.plot(Q.longitude, sel_(Q["fsns"]).sel(year=2000, param=p))
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(p)

src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[210,270], xlines=[210,270])

